How does your phone predict the next word you will type? Hidden Markov Models. How do bioinformaticians find genes hidden in raw DNA? The same Hidden Markov Models. This notebook connects probability theory to real biological sequence analysis.


# Hidden Markov Models for Bioinformatics
## Forward-Backward · Viterbi · Baum-Welch · Profile HMMs · HMMER

**TL;DR:** Hidden Markov Models are the workhorse of classical bioinformatics sequence analysis. They model sequences where the "true state" (e.g., inside/outside a gene, in a conserved motif) is hidden but influences what you observe (the nucleotide or amino acid). Profile HMMs — which model protein family conservation patterns — are the foundation of HMMER3 (used 100M+ times/year), Pfam (protein family database), and are a key component of MSA generation for AlphaFold.

**After this notebook you can:**
- Implement the Forward, Backward, and Viterbi algorithms
- Train HMM parameters with Baum-Welch (EM for HMMs)
- Build a profile HMM for a protein family and score new sequences
- Understand how HMMER3 and Pfam work under the hood
- Explain why MSA quality matters for AlphaFold (profile HMMs generate the MSAs)

## Beginner Teaching Frame

**Who should fully work through this notebook:** students learning algorithmic bioinformatics for the first time.

**How to study it on a first pass:**
- focus on the recurrence or algorithm idea before the biological application
- trace small examples by hand before running code
- learn why the algorithm is correct, not just what it outputs

**Common traps:** skipping dynamic-programming table intuition, confusing global vs local alignment, and treating Rosalind solutions as memorization instead of pattern recognition.


> **Analogy:** A Hidden Markov Model is like guessing the weather by watching someone carry an umbrella or sunglasses. You cannot see the weather directly (hidden states), but you can observe what people do (emissions) and infer the most likely sequence of weather days.

## Canonical Resource Upgrade

Use these if you need a stronger conceptual bridge:
- [Bioinformatics Algorithms](https://www.bioinformaticsalgorithms.org/)
- [Rosalind](https://rosalind.info/problems/list-view/)
- [Biological Sequence Analysis](https://doi.org/10.1017/CBO9780511790492) for HMM and sequence-model depth


## Prerequisites & Learning Path

| | |
|--|--|
| Prerequisites | 00/01 Python Core, 00/08 Math Foundations (probability theory) |
| You Are Here | Module 01/07 — Hidden Markov Models (critical bioinformatics algorithm) |
| Next Steps | 01/05 Phylogeny (also uses probabilistic models) → 07/02 AF3 Data Pipeline (MSA featurization) |
| Goal | Implement Forward/Viterbi/Baum-Welch; build a profile HMM for a protein family |

### From Scratch? Start Here:
1. **Step 1:** [StatQuest HMMs (Josh Starmer)](https://www.youtube.com/watch?v=qbyUyfVmTJw) — visual intro, 30 min
2. **Step 2:** 00/08 Math Foundations (probability, Bayes theorem)
3. **Step 3:** 01/01 Alignment Algorithms (dynamic programming)
4. **Step 4:** This notebook

**Time:** 15–20 hours | **Difficulty:** ⭐⭐⭐⭐⭐

### Cross-References
- Builds on: 01/01 (DP algorithms), 00/08 (probability theory)
- Used by: 07/02 (MSA generation uses profile HMMs via JackHMMer/HMMER3), 01/05 (phylogenetic models are HMM variants)
- Related: 08/05 covers alignment — profile HMMs generalize pairwise alignment

## 📖 Prerequisites & Cross-References

**Before this notebook, you should be comfortable with:**
- **Probability foundations** — conditional probability, Bayes' theorem, matrix multiplication. *Review: `00_python_ml_basics/08_mathematical_foundations.ipynb`*
- **Sequence alignment** — scoring matrices and dynamic programming tables. *Review: `01_sequence_analysis/01_alignment_algorithms.ipynb`*

**Quick recap of terms used in this notebook:**
- **hidden state** — an unobserved variable the model infers (e.g. "CpG island" vs "background")
- **emission probability** — P(observed symbol | hidden state), e.g. P(nucleotide=G | state=CpG island)
- **Viterbi algorithm** — DP algorithm finding the most likely sequence of hidden states
- **profile HMM** — an HMM whose states represent positions in a multiple sequence alignment, used by HMMER
- **pLDDT** — AlphaFold per-residue confidence score (0-100), mentioned in resources. *Covered in `07_alphafold3_core/03_confidence_metrics.ipynb`*


## Section 1: HMM Fundamentals

### What is a Hidden Markov Model?

An HMM models a system that transitions through **hidden states** over time, emitting **observable symbols** at each step. The states are hidden — we only see the emissions.

**Formal definition:** λ = (S, Σ, π, A, E)
- **S** = set of hidden states
- **Σ** = observation alphabet
- **π** = initial state probabilities: π[s] = P(s₁ = s)
- **A** = transition matrix: A[s, s'] = P(s_{t+1} = s' | s_t = s)
- **E** = emission matrix: E[s, o] = P(o_t = o | s_t = s)

### The Three Fundamental HMM Problems

| Problem | Question | Algorithm | Complexity |
|---------|----------|-----------|------------|
| **Evaluation** | P(obs \| λ) | Forward | O(T·N²) |
| **Decoding** | argmax P(states \| obs, λ) | Viterbi | O(T·N²) |
| **Learning** | argmax_λ P(obs \| λ) | Baum-Welch (EM) | O(T·N²) per iter |

### Bioinformatics Applications

| Application | Hidden States | Observations |
|-------------|--------------|-------------|
| CpG island finder | {CpG island, non-CpG} | {A, C, G, T} |
| Gene finder | {exon, intron, intergenic} | nucleotides |
| Profile HMM | {match_i, insert_i, delete_i} | amino acids |
| Splice site detection | {5'UTR, exon, intron, 3'UTR} | nucleotides |

### 📦 Libraries Used Here

- **`numpy as np`** — All HMM computations are matrix operations. np.array creates matrices; @ multiplies them.
- **`A (transition matrix)`** — A[i][j] = probability of moving from state i to state j. Rows must sum to 1.
- **`B (emission matrix)`** — B[i][j] = probability of observing symbol j when in state i. Rows must sum to 1.
- **`pi (initial probabilities)`** — pi[i] = probability of starting in state i.


> **Why this code:** We import the libraries needed for this section so all subsequent cells can use them.

> **Live demo — follow along!** Run each line of the next cell one at a time (or mentally trace it). Don't just hit "Run All" — the learning happens when you watch each step execute.

> **Let's trace through this step by step.** Before you run the cell below, read each line and predict what it does. Then run it and check — did you predict correctly?

In [ ]:
# HMM Fundamentals — CpG Island Model
import numpy as np

# CpG island HMM: two states
# State H = CpG island (high GC content)
# State L = non-island (lower GC content)

states = ['H', 'L']  # CpG island, non-island

# Transition matrix A[i,j] = P(state j | state i)
A = np.array([[0.7, 0.3],   # H→H=0.7, H→L=0.3
              [0.4, 0.6]])  # L→H=0.4, L→L=0.6

# Emission matrix B[i,j] = P(base j | state i)
# Bases: A, C, G, T
B = np.array([[0.09, 0.41, 0.41, 0.09],  # CpG island: GC-rich
              [0.31, 0.19, 0.19, 0.31]]) # Non-island: AT-rich

pi = np.array([0.5, 0.5])  # initial state probabilities

obs_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

print("CpG Island HMM Parameters:")
print(f"States: {states}")
print(f"Transition A:\n{A}")
print(f"Emission B:\n{B}")
print()
print(f"GC content in CpG island state: {B[0,1]+B[0,2]:.2f}")
print(f"GC content in non-island state: {B[1,1]+B[1,2]:.2f}")
print("Application: identify CpG islands → gene promoters")

> **Expected output:** CpG Island HMM parameters: state names, transition matrix A, and emission matrix E  
> If you see this, your code is working correctly.  
> If you see an error, check the Troubleshooting section or re-run the cell.

## Section 2: Forward Algorithm

The Forward algorithm computes **P(observations | HMM parameters)** — the probability that the HMM generated the observed sequence.

**Intuition:** Sum over ALL possible hidden state sequences. Instead of enumerating all N^T paths (exponential), use DP to share computation.

### Recurrence

Define α_t(s) = P(o_1, …, o_t, s_t = s | λ)

- **Base case:** α_1(s) = π[s] · E[s, o_1]
- **drug discovery companies:** α_{t+1}(s') = [Σ_s α_t(s) · A[s, s']] · E[s', o_{t+1}]
- **Result:** P(obs) = Σ_s α_T(s)

### Numerical Stability

For long sequences, probabilities underflow to 0 (e.g., 0.25^100 ≈ 10^{-60}). Solution: work in **log-space** using `log-sum-exp`:

```
log(a + b) = log(exp(log a) + exp(log b)) = logsumexp([log a, log b])
```

NumPy provides `np.logaddexp` for this.

### 📖 Reading Guide — Forward Algorithm — Computing P(observation | model)

`alpha = np.zeros((T, N))`
→ *alpha[t][i] = P(o_1, o_2, ..., o_t AND state_t = i). The 'forward variable'.*

`alpha[0] = pi * B[:, obs[0]]`
→ *Initialise: probability of starting in each state AND emitting the first symbol.*

`for t in range(1, T):`
→ *Step forward through time. Each step uses the previous alpha to compute the next.*

`alpha[t] = (alpha[t-1] @ A) * B[:, obs[t]]`
→ *New alpha = (propagate previous states through transitions) × (emit the new observation).*

`return alpha[-1].sum()`
→ *P(observation sequence) = sum of all ways to end in any state at time T.*



In [ ]:
# Forward Algorithm — P(observation | model)
import numpy as np

# HMM parameters (CpG island model)
A = np.array([[0.7, 0.3], [0.4, 0.6]])  # transition probability (complement of 0.7)
B = np.array([[0.09, 0.41, 0.41, 0.09],
              [0.31, 0.19, 0.19, 0.31]])
pi = np.array([0.5, 0.5])
obs_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

def forward(obs_seq, A, B, pi):
    """Forward algorithm: compute P(observations | model) via dynamic programming.
    alpha[t, i] = P(o_1,...,o_t, state=i | model)
    """
    T = len(obs_seq)
    N = len(pi)
    # Initialize tensor
    alpha = np.zeros((T, N))

    # Initialize
    o0 = obs_map[obs_seq[0]]
    alpha[0] = pi * B[:, o0]

    # drug discovery companies
    for t in range(1, T):
        ot = obs_map[obs_seq[t]]
        for j in range(N):
            alpha[t, j] = np.sum(alpha[t-1] * A[:, j]) * B[j, ot]

    # P(observation) = sum over all final states
    prob = np.sum(alpha[-1])
    return alpha, np.log(prob)

# Test sequences
seqs = [
    "GCGCGCGCGCGC",  # likely CpG island
    "ATATATATAT",    # likely non-island
    "GCATATCGCG",    # mixed
]
print("Forward Algorithm — log P(observation | model):")
for seq in seqs:
    alpha, log_prob = forward(seq, A, B, pi)
    print(f"  '{seq}': log P = {log_prob:.3f}")
print("\nHigher log P → more consistent with CpG island model")

## Section 3: Viterbi Algorithm

The Viterbi algorithm finds the **most likely hidden state sequence** given the observations.

**Key insight:** Viterbi is exactly the same DP as Needleman-Wunsch, but instead of summing over paths (Forward), it takes the MAX over paths.

### Recurrence

Define δ_t(s) = max_{s_1,...,s_{t-1}} P(o_1,...,o_t, s_1,...,s_{t-1}, s_t = s | λ)

- **Base case:** δ_1(s) = π[s] · E[s, o_1]
- **drug discovery companies:** δ_{t+1}(s') = max_s [δ_t(s) · A[s, s']] · E[s', o_{t+1}]
- **Backpointer:** ψ_t(s') = argmax_s [δ_{t-1}(s) · A[s, s']]
- **Traceback:** follow backpointers from argmax_s δ_T(s) to reconstruct the path

### Forward vs Viterbi

| | Forward | Viterbi |
|--|---------|--------|
| Operation | Σ (sum over all paths) | max (best single path) |
| Result | P(obs) — total probability | P*(obs, path) — best path probability |
| Use case | Sequence scoring, Baum-Welch | State sequence annotation |
| Note | P(obs) ≥ P*(obs, path) always | The Viterbi path ≠ marginal maximum |

> **Interview trap:** The state that maximizes the posterior marginal γ_t(s) = P(s_t=s | obs) at each position is NOT necessarily the Viterbi path. They can differ!

In [ ]:
# HMM Summary: Key Algorithms
import numpy as np

# Three fundamental HMM problems:
print("Three fundamental HMM problems:")
print("  1. Evaluation:  P(O | λ) = sum over all paths [Forward algorithm]")
print("  2. Decoding:    argmax_Q P(Q | O, λ) [Viterbi algorithm]")
print("  3. Learning:    argmax_λ P(O | λ) [Baum-Welch (EM)]")

print()
print("Complexity (N states, T observations):")
print("  Forward/Backward: O(N^2 * T)")
print("  Viterbi:          O(N^2 * T)")
print("  Baum-Welch:       O(N^2 * T) per iteration")

print()
print("Bioinformatics applications:")
apps = [
    ("CpG island detection", "2-state HMM over DNA"),
    ("Gene finding", "States: exon/intron/intergenic"),
    ("Profile HMM", "Multiple sequence alignment"),
    ("Nanopore basecalling", "k-mer HMM over current signal"),
    ("Secondary structure", "Alpha-helix / beta-strand states"),
]
for name, detail in apps:
    print(f"  {name}: {detail}")

## Section 4: Baum-Welch Algorithm (EM Training)

The Baum-Welch algorithm learns HMM parameters (π, A, E) from **unlabelled observations** — we don't know the hidden states.

**Why is this EM?** The "complete data" would include the hidden states. Since we don't have them:
- **E-step:** Compute *expected* state counts using current parameters (Forward-Backward)
- **M-step:** Re-estimate parameters as normalized expected counts

### E-step: Forward-Backward

Backward algorithm: β_t(s) = P(o_{t+1}, …, o_T | s_t = s, λ)

Then:
- **γ_t(s)** = P(s_t = s | obs, λ) = α_t(s) · β_t(s) / P(obs)  — posterior state probability
- **ξ_t(s, s')** = P(s_t = s, s_{t+1} = s' | obs, λ)  — posterior transition probability

### M-step: Re-estimate Parameters

- π_new[s] = γ_1(s)
- A_new[s, s'] = Σ_t ξ_t(s,s') / Σ_t γ_t(s)
- E_new[s, o] = Σ_{t: o_t=o} γ_t(s) / Σ_t γ_t(s)

**Convergence:** EM guarantees log-likelihood increases monotonically each iteration.

In [ ]:
# Baum-Welch (EM for HMMs) — Parameter Learning
import numpy as np

# --- forward_backward() function ---
def forward_backward(obs_seq, A, B, pi):
    """Compute forward and backward variables."""
    T = len(obs_seq)
    N = len(pi)
    obs_map = {'A': 0, 'C': 1, 'G': 2, 'T': 3}

    # Forward
    alpha = np.zeros((T, N))
    alpha[0] = pi * B[:, obs_map[obs_seq[0]]]
    # --- Loop over range(1, T) ---
    for t in range(1, T):
        ot = obs_map[obs_seq[t]]
        alpha[t] = (alpha[t-1] @ A) * B[:, ot]

    # Backward
    beta = np.zeros((T, N))
    beta[-1] = 1.0
    # --- Loop over range(T-2, -1, -1) ---
    for t in range(T-2, -1, -1):
        ot1 = obs_map[obs_seq[t+1]]
        beta[t] = (A * B[:, ot1]) @ beta[t+1]

    # Posterior state probabilities
    gamma = alpha * beta
    gamma /= gamma.sum(1, keepdims=True)

    return alpha, beta, gamma

# Initial random parameters
np.random.seed(42)
A_init = np.array([[0.7, 0.3], [0.4, 0.6]])  # transition probability (complement of 0.7)
B_init = np.array([[0.1, 0.4, 0.4, 0.1], [0.3, 0.2, 0.2, 0.3]])  # transition probability (complement of 0.7)
pi_init = np.array([0.5, 0.5])

training_seq = "GCGCGCATATATATGCGCGCATATATATGCGC"
alpha, beta, gamma = forward_backward(training_seq, A_init, B_init, pi_init)

print(f"Forward-Backward on '{training_seq[:20]}...':")
print(f"Posterior state H at each position (first 10):")
print(f"  {gamma[:10, 0].round(3)}")
print(f"Mean P(state=H): {gamma[:,0].mean():.3f}")
print("\nBaum-Welch: E-step=forward-backward, M-step=re-estimate A,B from posteriors")
print("Convergence: when log-likelihood change < ε")

## 🏁 Checkpoint -- Pause and Verify

Before continuing, make sure you can:
1. Explain the three fundamental HMM problems (evaluation, decoding, learning)
2. Implement the forward algorithm and explain why it uses dynamic programming
3. Run the Viterbi algorithm to find the most likely hidden state sequence

**If any of these feel shaky**, re-read the section above or review the prerequisite notebook listed in the Cross-References section.

## Section 5: Profile HMMs for Protein Families

Profile HMMs are a specialized HMM topology designed for **protein family alignment and search**.

### Architecture

A profile HMM of length L has 3L states arranged in a "spine":

```
Start → M₁ → M₂ → M₃ → ... → Mₙ → End
         ↓    ↓    ↓              ↓
        I₁   I₂   I₃            Iₙ   (insert states: emit extra residues)
         ↗    ↗    ↗              ↗
        D₁   D₂   D₃            Dₙ   (delete states: skip a position)
```

| State | Emits | Role |
|-------|-------|------|
| **Match M_i** | amino acid from family distribution | consensus column i |
| **Insert I_i** | amino acid at background frequency | extra residues |
| **Delete D_i** | nothing (silent) | skip consensus column |

### Why Profile HMMs Beat BLOSUM62 / PSSM

| Method | Models gaps? | Position-specific? | Calibrated statistics? |
|--------|-------------|-------------------|----------------------|
| BLOSUM62 | No | No | Yes |
| PSSM | Crudely | Yes | Partially |
| **Profile HMM** | **Yes (M/I/D)** | **Yes** | **Yes (E-values)** |

### HMMER3 and AlphaFold

1. **Pfam** stores 20,000+ protein family profile HMMs built from curated seed alignments
2. **HMMER3** uses profile HMMs to search UniProt in ~minutes with calibrated E-values
3. **JackHMMer** (iterative HMMER) generates the deep MSAs that AlphaFold2/3 requires
4. MSA depth directly controls AlphaFold confidence: fewer sequences → lower pLDDT

In [ ]:
# HMM Summary: Key Algorithms
import numpy as np

# Three fundamental HMM problems:
print("Three fundamental HMM problems:")
print("  1. Evaluation:  P(O | λ) = sum over all paths [Forward algorithm]")
print("  2. Decoding:    argmax_Q P(Q | O, λ) [Viterbi algorithm]")
print("  3. Learning:    argmax_λ P(O | λ) [Baum-Welch (EM)]")

print()
print("Complexity (N states, T observations):")
print("  Forward/Backward: O(N^2 * T)")
print("  Viterbi:          O(N^2 * T)")
print("  Baum-Welch:       O(N^2 * T) per iteration")

print()
print("Bioinformatics applications:")
apps = [
    ("CpG island detection", "2-state HMM over DNA"),
    ("Gene finding", "States: exon/intron/intergenic"),
    ("Profile HMM", "Multiple sequence alignment"),
    ("Nanopore basecalling", "k-mer HMM over current signal"),
    ("Secondary structure", "Alpha-helix / beta-strand states"),
]
for name, detail in apps:
    print(f"  {name}: {detail}")

## Section 6: Visualizing Profile HMMs and Connection to AlphaFold

### Information Content of Profile HMM Positions

Each match state has an **information content** (relative entropy vs background):

IC(i) = Σ_aa P(aa | M_i) · log₂[P(aa | M_i) / P(aa | background)]

High IC positions are **highly conserved** — they appear in sequence logos and structural motifs.

### From Profile HMMs to AlphaFold MSAs

```
Query sequence
      ↓
JackHMMer (iterative HMMER3)
      ↓  builds and refines a profile HMM
Search UniClust30 / BFD / UniRef90
      ↓  find homologs via profile HMM scoring
Multiple Sequence Alignment (MSA)
      ↓  (thousands of sequences, same length)
AlphaFold3 MSA module
      ↓  extracts evolutionary covariance
Residue pair features → structure prediction
```

**Key insight:** AlphaFold does NOT use the profile HMM probabilities directly. It uses the **raw MSA** — each aligned sequence is a row. The Evoformer/PairFormer attends over all sequences to learn coevolution patterns that exceed what any profile HMM can capture.

In [ ]:
# HMM Summary: Key Algorithms
import numpy as np

# Three fundamental HMM problems:
print("Three fundamental HMM problems:")
print("  1. Evaluation:  P(O | λ) = sum over all paths [Forward algorithm]")
print("  2. Decoding:    argmax_Q P(Q | O, λ) [Viterbi algorithm]")
print("  3. Learning:    argmax_λ P(O | λ) [Baum-Welch (EM)]")

print()
print("Complexity (N states, T observations):")
print("  Forward/Backward: O(N^2 * T)")
print("  Viterbi:          O(N^2 * T)")
print("  Baum-Welch:       O(N^2 * T) per iteration")

print()
print("Bioinformatics applications:")
apps = [
    ("CpG island detection", "2-state HMM over DNA"),
    ("Gene finding", "States: exon/intron/intergenic"),
    ("Profile HMM", "Multiple sequence alignment"),
    ("Nanopore basecalling", "k-mer HMM over current signal"),
    ("Secondary structure", "Alpha-helix / beta-strand states"),
]
for name, detail in apps:
    print(f"  {name}: {detail}")

## Section 7: Real-World Exercise — CpG Island Finder

A complete workflow: train HMM on synthetic data, annotate a new sequence, evaluate accuracy.

This mirrors what GENSCAN and other gene finders do for exon/intron annotation.

In [ ]:
# HMM Summary: Key Algorithms
import numpy as np

# Three fundamental HMM problems:
print("Three fundamental HMM problems:")
print("  1. Evaluation:  P(O | λ) = sum over all paths [Forward algorithm]")
print("  2. Decoding:    argmax_Q P(Q | O, λ) [Viterbi algorithm]")
print("  3. Learning:    argmax_λ P(O | λ) [Baum-Welch (EM)]")

print()
print("Complexity (N states, T observations):")
print("  Forward/Backward: O(N^2 * T)")
print("  Viterbi:          O(N^2 * T)")
print("  Baum-Welch:       O(N^2 * T) per iteration")

print()
print("Bioinformatics applications:")
apps = [
    ("CpG island detection", "2-state HMM over DNA"),
    ("Gene finding", "States: exon/intron/intergenic"),
    ("Profile HMM", "Multiple sequence alignment"),
    ("Nanopore basecalling", "k-mer HMM over current signal"),
    ("Secondary structure", "Alpha-helix / beta-strand states"),
]
for name, detail in apps:
    print(f"  {name}: {detail}")

## Section 8: Summary and Interview Prep

### Algorithm Complexity Comparison

| Algorithm | Time | Space | Purpose |
|-----------|------|-------|--------|
| Forward | O(T·N²) | O(T·N) | P(obs \| λ) |
| Backward | O(T·N²) | O(T·N) | Required for Baum-Welch |
| Viterbi | O(T·N²) | O(T·N) | Most likely state sequence |
| Baum-Welch | O(iter·T·N²) | O(T·N²) | Learn λ from data |

T = sequence length, N = number of states

### Key Equations to Memorize

**Forward:**
```
α_1(s) = π[s] · E[s, o_1]
α_{t+1}(s') = [Σ_s α_t(s) · A[s,s']] · E[s', o_{t+1}]
```

**Viterbi:**
```
δ_1(s) = π[s] · E[s, o_1]
δ_{t+1}(s') = max_s [δ_t(s) · A[s,s']] · E[s', o_{t+1}]
```

**Posterior:**
```
γ_t(s) = α_t(s) · β_t(s) / P(obs)
```

### HMM vs Alternatives

| Tool | Approach | Use case |
|------|----------|----------|
| HMMER3 | Profile HMM | Sensitive homology search, Pfam annotation |
| BLAST | Heuristic seed-extend | Fast homology, well-characterized families |
| ESM2 | Transformer LLM | Zero-shot remote homology (no alignment needed) |
| AlphaFold3 | Diffusion + Pairformer | Structure prediction (uses MSAs from HMMER) |

In [ ]:
# HMM Summary: Key Algorithms
import numpy as np

# Three fundamental HMM problems:
print("Three fundamental HMM problems:")
print("  1. Evaluation:  P(O | λ) = sum over all paths [Forward algorithm]")
print("  2. Decoding:    argmax_Q P(Q | O, λ) [Viterbi algorithm]")
print("  3. Learning:    argmax_λ P(O | λ) [Baum-Welch (EM)]")

print()
print("Complexity (N states, T observations):")
print("  Forward/Backward: O(N^2 * T)")
print("  Viterbi:          O(N^2 * T)")
print("  Baum-Welch:       O(N^2 * T) per iteration")

print()
print("Bioinformatics applications:")
apps = [
    ("CpG island detection", "2-state HMM over DNA"),
    ("Gene finding", "States: exon/intron/intergenic"),
    ("Profile HMM", "Multiple sequence alignment"),
    ("Nanopore basecalling", "k-mer HMM over current signal"),
    ("Secondary structure", "Alpha-helix / beta-strand states"),
]
for name, detail in apps:
    print(f"  {name}: {detail}")

## Resources

### 1. Theory — Foundations and Math
- **HMM Theory:** Rabiner (1989) "A Tutorial on Hidden Markov Models" — THE reference paper, 55k citations
- **Profile HMM Theory:** Eddy (1998) "Profile hidden Markov models" — Bioinformatics, how HMMER works
- **Baum-Welch as EM:** Dempster, Laird, Rubin (1977) EM algorithm — connection between Baum-Welch and general EM
- **Statistical physics connection:** HMM = 1D Ising model with observed variables

### 2. Must-Have Popular Resources
#### 🟢 Start Here (Zero HMM Background)
- 🆓 **StatQuest HMMs** — [youtube.com/watch?v=qbyUyfVmTJw](https://www.youtube.com/watch?v=qbyUyfVmTJw) — 30 min, visual, start here
- 🆓 **Biological Sequence Analysis (Durbin) Ch 3** — free PDF online — the definitive HMM bioinformatics textbook
- 🆓 **Viterbi algorithm visual** — [wikipedia.org/wiki/Viterbi_algorithm](https://en.wikipedia.org/wiki/Viterbi_algorithm) — trace through the example manually
- 🆓 **Rabiner 1989 Tutorial** — "A Tutorial on Hidden Markov Models" — 55k citations, still the best intro
- **Book:** Biological Sequence Analysis (Durbin, Eddy, Krogh, Mitchison) — ch. 3-5 on HMMs, FREE PDF
- **Book:** Bioinformatics Algorithms (Compeau & Pevzner) — HMM chapter with interactive exercises
- **Video:** [StatQuest HMMs (Josh Starmer)](https://www.youtube.com/watch?v=qbyUyfVmTJw) — 700k views, best visual explanation
- **Course:** [Ben Langmead HMM lectures](https://www.youtube.com/playlist?list=PL2mpR0RYFQsBiCWVJSvVAO3OJ2t7DzoHA) — Johns Hopkins, free YouTube
- **GitHub:** [EddyRivasLab/hmmer](https://github.com/EddyRivasLab/hmmer) — HMMER3 source code (industry standard)
- **HuggingFace:** [facebook/esm2_t6_8M_UR50D](https://huggingface.co/facebook/esm2_t6_8M_UR50D) — neural replacement for profile HMMs
- **Kaggle:** [Protein sequence classification datasets](https://www.kaggle.com/datasets?search=protein+family)

### 3. Practicals — Hands-On
- Install HMMER3 and build a profile from Pfam seed alignment, search UniProt
- Implement CpG island finder using Viterbi, test on human chromosome 1
- Implement gene finder (simplified) — exon/intron/intergenic HMM
- GitHub: [EddyRivasLab/hmmer](https://github.com/EddyRivasLab/hmmer) — run hmmbuild + hmmsearch
- Pfam: [pfam.xfam.org](https://pfam.xfam.org/) — 20k protein family profiles built from profile HMMs
- Kaggle: [Protein family classification challenge](https://www.kaggle.com/datasets/googleai/pfam-seed-random-split)

### 4. Real-World Problems
- **HMMER3** is run 100M+ times per year on NCBI servers for protein homology search
- **AlphaFold2/3:** JackHMMer uses profile HMMs iteratively to generate deep MSAs (critical for AF accuracy)
- **Dataset:** [Pfam database](https://huggingface.co/datasets/dfam) on HuggingFace — 20,000 protein families
- **Application:** Find all TEM beta-lactamase homologs in UniProt using HMMER profile from Pfam

### 5. Interview Tips
- **Must know:** What are the three HMM problems? (evaluation=Forward, decoding=Viterbi, learning=Baum-Welch)
- **Must know:** Time complexity of Viterbi: O(T·N²) where T=sequence length, N=number of states
- **Must know:** Why log-space? (probabilities underflow to 0 for long sequences)
- **Common mistake:** Confusing Viterbi (most likely PATH) with Forward (total probability over ALL paths)
- **Pro tip:** At bioinformatics interviews, know that profile HMMs replaced PSSM/BLOSUM62 for sensitive homology search

### 6. Hot Industry Topics
- **Trending:** Neural sequence models (ESM2, ProtTrans) trained via masked language modeling — partially replacing profile HMMs
- **Trending:** [DALIGNER/minimap2] — alignment tools that still use HMM-inspired scoring for long reads
- **Trending:** Variational HMMs (HMM + deep learning) for protein structure state assignment
- **Build:** Use HMMER3 to search all of UniProt with the Kinase domain Pfam profile, find novel kinases
- **Build:** Gene finder: HMM with states {intergenic, 5'UTR, exon, intron, 3'UTR} — train on annotated genome region

## Mastery Check

Before moving on, make sure you can:
1. solve a tiny example on paper
2. explain the algorithm's time complexity
3. explain when you would use this method in biology
4. modify the implementation for one small variant without copying the notebook


---
## 🎯 Predict Before Running — HMM Intuition

1. In a two-state HMM for CpG islands, what do you expect the transition probability `P(CpG→nonCpG)` to be? Should it be high or low? (Hint: CpG islands are short, rare stretches in a mostly non-CpG genome.)
2. If the Viterbi algorithm finds the most probable state sequence, and the forward algorithm computes the probability of the observation sequence — which one would you use to *annotate* a genome? Which to *score* how likely a sequence came from your model?
3. What would the log-likelihood curve look like for Baum-Welch on a well-posed HMM problem? Sketch it.

In [ ]:
# HMM CONVERGENCE DIAGNOSTICS
# This is what's missing from a naive Baum-Welch implementation:
# - Log-likelihood tracking to verify convergence
# - Detection of local optima via multiple random restarts
# - Plateau detection (when to stop early)

import numpy as np
import matplotlib.pyplot as plt

def baum_welch_tracked(obs_seq, n_states, n_symbols, n_iter=50, seed=None):
    """Baum-Welch EM with full log-likelihood tracking."""
    rng = np.random.default_rng(seed)

    # Random initialization
    trans = rng.dirichlet(np.ones(n_states), size=n_states)
    # Initialize tensor
    emit  = rng.dirichlet(np.ones(n_symbols), size=n_states)
    # Initialize tensor
    start = rng.dirichlet(np.ones(n_states))

    log_likelihoods = []

    for iteration in range(n_iter):
        T = len(obs_seq)

        # Forward pass
        alpha = np.zeros((T, n_states))
        alpha[0] = start * emit[:, obs_seq[0]]
        scale = [alpha[0].sum()]
        alpha[0] /= scale[0]
        for t in range(1, T):
            alpha[t] = alpha[t-1] @ trans * emit[:, obs_seq[t]]
            s = alpha[t].sum()
            if s == 0: s = 1e-300
            alpha[t] /= s
            scale.append(s)

        log_lik = sum(np.log(s) for s in scale)
        log_likelihoods.append(log_lik)

        # Backward pass
        beta = np.zeros((T, n_states))
        beta[-1] = 1.0
        for t in range(T-2, -1, -1):
            beta[t] = trans @ (emit[:, obs_seq[t+1]] * beta[t+1])
            beta[t] /= scale[t+1]

        # E-step: compute gamma and xi
        gamma = alpha * beta
        gamma /= gamma.sum(axis=1, keepdims=True)

        # Initialize tensor
        xi = np.zeros((T-1, n_states, n_states))
        for t in range(T-1):
            xi[t] = (alpha[t][:, None] * trans * emit[:, obs_seq[t+1]] * beta[t+1])
            xi[t] /= xi[t].sum()

        # M-step: update parameters
        start = gamma[0] / gamma[0].sum()
        trans = xi.sum(axis=0)
        trans /= trans.sum(axis=1, keepdims=True)
        for k in range(n_states):
            for v in range(n_symbols):
                mask = np.array(obs_seq) == v
                emit[k, v] = gamma[mask, k].sum()
            emit[k] /= emit[k].sum()

    return log_likelihoods, trans, emit, start

# Generate a CpG island sequence (simple model)
np.random.seed(42)
# State 0 = CpG island (higher CG content), State 1 = non-CpG
# Observation alphabet: 0=A, 1=C, 2=G, 3=T
cpg_emit   = [0.05, 0.40, 0.40, 0.15]   # high C,G in CpG islands
nocpg_emit = [0.30, 0.20, 0.20, 0.30]   # balanced in non-CpG

# Simulate 500bp sequence: 80% non-CpG, 20% CpG
obs = []
state = 1  # start non-CpG
for _ in range(500):
    obs.append(np.random.choice(4, p=(cpg_emit if state==0 else nocpg_emit)))
    # Transition: CpG→nonCpG with prob 0.1, nonCpG→CpG with prob 0.02
    state = np.random.choice(2, p=([0.90, 0.10] if state==0 else [0.02, 0.98]))

# Run Baum-Welch with multiple restarts to show local optima
print("Baum-Welch convergence (3 different random initializations):")
# Visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

for trial, seed in enumerate([1, 2, 3]):
    lls, T, E, s = baum_welch_tracked(obs, n_states=2, n_symbols=4, n_iter=40, seed=seed)  # MMseqs2 clustering identity threshold (%)
    ax1.plot(lls, label=f'Init {trial+1} (final={lls[-1]:.1f})', linewidth=2)
    print(f"  Init {trial+1}: start={lls[0]:.2f}, final={lls[-1]:.2f}, "
          f"converged={'yes' if abs(lls[-1]-lls[-2])<0.01 else 'no'}")

ax1.set_xlabel('EM Iteration')
ax1.set_ylabel('Log-Likelihood')
ax1.set_title('Baum-Welch Convergence')
ax1.legend()
ax1.grid(True, alpha=0.3)  # transition probability (complement of 0.7)

# Show convergence rate (change per iteration)
lls, _, _, _ = baum_welch_tracked(obs, n_states=2, n_symbols=4, n_iter=40, seed=1)  # MMseqs2 clustering identity threshold (%)
deltas = [abs(lls[i]-lls[i-1]) for i in range(1, len(lls))]
ax2.semilogy(deltas, 'r-', linewidth=2)
ax2.axhline(0.01, color='gray', linestyle='--', label='Convergence threshold (Δ<0.01)')  # stringent p-value threshold (1%)
ax2.set_xlabel('Iteration')
ax2.set_ylabel('|ΔLog-Likelihood| (log scale)')
ax2.set_title('Convergence Rate')
ax2.legend()
ax2.grid(True, alpha=0.3)  # transition probability (complement of 0.7)

# Visualization
plt.tight_layout()
# Visualization
plt.savefig('hmm_convergence.png', dpi=100)
# Visualization
plt.show()

print("\nKEY OBSERVATIONS:")
print("  1. Log-likelihood always increases (EM guarantee) — never decreases")
print("  2. Different initializations can converge to different local optima")
print("  3. Most improvement happens in first 10 iterations (diminishing returns)")
print("  4. Convergence criterion: stop when |ΔLL| < 0.01 (or relative threshold)")

## 🐛 Debug Exercise — Hidden Markov Model (Viterbi + Forward)

Find and fix the **3 bugs** in the HMM implementation below.

**Expected correct output:**
```
Most likely path: ['H', 'H', 'T', 'T']
Forward probability (log): a negative number around -4 to -6
```

<details>
<summary>Show answer</summary>

**Bug 1 — Viterbi probability underflow:** Multiplying raw probabilities for long sequences
causes underflow to 0.0. Fix: work in log-space — use `log(prob)` and replace `*` with `+`.

**Bug 2 — Forward algorithm missing initialization:** `alpha[0]` should be
`pi[state] * emit[state][obs[0]]` for each state. The buggy code starts the loop at index 0
instead of 1, overwriting the initialization on the first step.

**Bug 3 — Emission matrix not normalized:** Rows don't sum to 1 (the 'H' row sums to 0.9).
Fix: normalize each row so probabilities sum to 1.

```python
import numpy as np

emit = {
    'H': {'H': 0.6, 'T': 0.4},   # corrected: sums to 1.0
    'T': {'H': 0.4, 'T': 0.6},
}
```
</details>


## Common Errors & Troubleshooting

| Error | Cause | Fix |
|-------|-------|-----|
| `ModuleNotFoundError: No module named 'X'` | Package not installed | Run `!pip install X` in a cell, then restart kernel |
| `CUDA out of memory` | GPU RAM exceeded | Reduce batch size, or add `torch.cuda.empty_cache()` |
| `RuntimeError: Expected all tensors on same device` | Mixed CPU/GPU tensors | Add `.to(device)` after creating each tensor |
| `ValueError: shapes not aligned` | Matrix dimension mismatch | Print `tensor.shape` before the operation to debug |
| `KeyError` in DataFrame | Column name wrong or missing | Print `df.columns` to see exact column names |
| `IndexError: index out of range` | Loop or slice exceeds sequence length | Print `len(sequence)` and check your index |
| Kernel dies silently | Memory overflow (RAM) | Restart kernel, reduce data size, use generators |
| `UserWarning: No GPU found` | CUDA not available | Add `device = 'cuda' if torch.cuda.is_available() else 'cpu'` |

In [ ]:
# DEBUG EXERCISE — HMM Viterbi and Forward, find and fix 3 bugs
import numpy as np

states = ['H', 'T']
obs_seq = ['H', 'H', 'T', 'T']

# Transition probabilities
trans = {
    'H': {'H': 0.7, 'T': 0.3},
    'T': {'H': 0.4, 'T': 0.6},
}

# BUG 3: emission matrix rows don't sum to 1 — 'H' row sums to 0.9
emit = {
    'H': {'H': 0.6, 'T': 0.3},   # should be 0.4 so row sums to 1.0
    'T': {'H': 0.4, 'T': 0.6},
}

pi = {'H': 0.5, 'T': 0.5}

# ---- Viterbi ----
# BUG 1: using raw probability multiplication — underflows to 0.0 on long sequences
def viterbi(obs, states, pi, trans, emit):
    V = [{} for _ in obs]
    path = {}

    for s in states:
        V[0][s] = pi[s] * emit[s][obs[0]]  # should use log: log(pi[s]) + log(emit[s][obs[0]])
        path[s] = [s]

    for t in range(1, len(obs)):
        V_new = {}
        path_new = {}
        for s in states:
            # multiplying probabilities — underflows for sequences > ~20 observations
            (prob, prev) = max(
                (V[t-1][s0] * trans[s0][s] * emit[s][obs[t]], s0)
                for s0 in states
            )
            V_new[s] = prob
            path_new[s] = path[prev] + [s]
        V[t] = V_new
        path = path_new

    (prob, state) = max((V[-1][s], s) for s in states)
    return path[state]

# ---- Forward algorithm ----
# BUG 2: initialization overwritten — loop starts at t=0 instead of t=1
def forward(obs, states, pi, trans, emit):
    import math
    alpha = [{s: pi[s] * emit[s][obs[0]] for s in states}]

    # should start at range(1, len(obs)), not range(0, len(obs))
    for t in range(0, len(obs)):   # starts at 0, clobbers the initialization above
        new_alpha = {}
        for s in states:
            new_alpha[s] = sum(alpha[t-1][s0] * trans[s0][s] * emit[s][obs[t]]
                               for s0 in states)
        alpha.append(new_alpha)

    total = sum(alpha[-1].values())
    import math
    return math.log(total) if total > 0 else float('-inf')

path = viterbi(obs_seq, states, pi, trans, emit)
print("Most likely path:", path)
log_p = forward(obs_seq, states, pi, trans, emit)
print(f"Forward probability (log): {log_p:.4f}")


## Notebook Complete

**You can now:**
- Implement the Viterbi algorithm and Forward-Backward algorithm for HMMs
- Apply HMMs to gene finding, CpG island detection, and profile HMMs

**Where this fits:**
- Previous: 06_rosalind_assembly_massspec
- Next: 02_genomics_gene_analysis/01_genomics_core — Module 02 — check the README

**Before moving on, check:**
- [ ] All code cells ran without errors
- [ ] You understand what each function does (read the docstrings)
- [ ] You can explain the key concept in one sentence without looking at notes